# FastAPI Test Console

Use this notebook to test `backend/api.py` without starting a separate Uvicorn server. It calls the FastAPI app in-process with `TestClient`, which is fast and keeps debugging simple.


In [ ]:
import importlib
from time import perf_counter

import pandas as pd
from fastapi.testclient import TestClient

import backend.utils as backend_utils
import backend.names.aliases as aliases
import backend.names.keywords as keywords
import backend.entities as entities
import backend.nba_client as nba_client
import backend.formatters as formatters
import backend.parsing as parsing
import backend.search_engine as search_engine
import backend.api as api

# Reload dependencies first so notebook kernels pick up local code edits.
for module in [
    backend_utils,
    aliases,
    keywords,
    entities,
    nba_client,
    formatters,
    parsing,
    search_engine,
    api,
]:
    importlib.reload(module)

client = TestClient(api.app)

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 60)


## Health Check

This confirms that `api.app` imports cleanly and the FastAPI route is registered.


In [ ]:
health_response = client.get("/health")

{
    "status_code": health_response.status_code,
    "body": health_response.json(),
}


## Query Request

Edit this payload to test your API behavior. Keep `use_play_by_play` false for the fast clutch approximation.


In [ ]:
payload = {
    "query": "harden clutch misses q4",
    "season": "2025-26",
    "limit": 10,
    "offset": 0,
    "use_play_by_play": False,
}

payload


## Run API Search

This calls `POST /query` through the same FastAPI request/response validation the frontend will use.


In [ ]:
started_at = perf_counter()
response = client.post("/query", json=payload)
notebook_latency_ms = round((perf_counter() - started_at) * 1000)

print("status:", response.status_code)
response.raise_for_status()
data = response.json()

summary = {
    "query": data["query"],
    "interpretation": data["interpretation"],
    "api_latency_ms": data["latency_ms"],
    "notebook_latency_ms": notebook_latency_ms,
    "raw_result_count": data["raw_result_count"],
    "filtered_result_count": data["filtered_result_count"],
    "returned_results": len(data["results"]),
    "warnings": data["warnings"],
    "query_params": data["query_params"],
}

summary


## Results Table

The API returns JSON records. This cell turns them back into a DataFrame just for notebook inspection.


In [ ]:
results_df = pd.DataFrame(data["results"])

display_columns = [
    "Game_ID",
    "Event_Index",
    "Game_Date",
    "Period",
    "Description",
    "Point_Change",
    "Score_Diff",
    "Score_Diff_After",
    "Video_Link",
    "Event_Link",
]
available_columns = [column for column in display_columns if column in results_df.columns]
results_df[available_columns]


## Pagination Check

Change `offset` to page through the filtered result set without changing the query. This still runs a fresh backend search today; later we can cache query responses if needed.


In [ ]:
next_page_payload = {**payload, "offset": payload["offset"] + payload["limit"]}
next_page_response = client.post("/query", json=next_page_payload)
next_page_response.raise_for_status()
next_page = next_page_response.json()

{
    "offset": next_page["offset"],
    "limit": next_page["limit"],
    "returned_results": len(next_page["results"]),
    "first_description": next_page["results"][0]["Description"] if next_page["results"] else None,
}


## Validation Check

This verifies FastAPI/Pydantic rejects invalid client input before it reaches the search engine.


In [ ]:
invalid_response = client.post("/query", json={**payload, "query": "", "limit": 500})

{
    "status_code": invalid_response.status_code,
    "body": invalid_response.json(),
}


## Optional Live Server Test

Set `RUN_LIVE_SERVER_TEST = True` only after starting Uvicorn in a terminal.

```powershell
.\.venv\Scripts\python.exe -m uvicorn backend.api:app --host 127.0.0.1 --port 8000 --reload
```


In [ ]:
RUN_LIVE_SERVER_TEST = False

if RUN_LIVE_SERVER_TEST:
    import requests

    live_response = requests.post("http://127.0.0.1:8000/query", json=payload, timeout=60)
    live_response.raise_for_status()
    live_data = live_response.json()
    live_summary = {
        "status_code": live_response.status_code,
        "interpretation": live_data["interpretation"],
        "filtered_result_count": live_data["filtered_result_count"],
        "returned_results": len(live_data["results"]),
    }
else:
    live_summary = "Set RUN_LIVE_SERVER_TEST = True after starting Uvicorn."

live_summary
